In [1]:
import os
import re
import time
import requests
import pandas as pd
from urllib.parse import urljoin
from PyPDF2 import PdfReader
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    InvalidSessionIdException, WebDriverException, TimeoutException, NoSuchElementException)

# === CONFIGURATION ===
CHROMEDRIVER_PATH = r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\chromedriver-win64\chromedriver.exe"
BASE_DIR = r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2"
PDF_DIR = os.path.join(BASE_DIR, "pdffiles")
CSV_DIR = os.path.join(BASE_DIR, "csvfiles")
DEBUG_DIR = os.path.join(BASE_DIR, "debug")
BASE_URL = "https://www.tunisieclearing.com/tc/fr/statistiques/historiquebulletin"
WAIT_TIMEOUT = 30


In [4]:
def setup_directories():
    os.makedirs(PDF_DIR, exist_ok=True)
    os.makedirs(CSV_DIR, exist_ok=True)
    os.makedirs(DEBUG_DIR, exist_ok=True)
    print("✅ Répertoires configurés")

def check_network():
    try:
        response = requests.head(BASE_URL, timeout=10)
        if response.status_code == 200:
            print("✅ Connexion réseau OK")
            return True
        print(f"⚠️ Échec de connexion: Statut {response.status_code}")
        return False
    except requests.RequestException as e:
        print(f"❌ Erreur réseau: {str(e)}")
        return False

def initialize_driver():
    print("🔍 Initialisation du navigateur...")
    options = Options()
    # options.add_argument("--headless=new")  # Décommente si tu veux le mode sans tête
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_experimental_option("prefs", {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.javascript": 1,
    })

    try:
        service = Service(executable_path=CHROMEDRIVER_PATH)
        driver = webdriver.Chrome(service=service, options=options)
        driver.set_page_load_timeout(60)
        driver.implicitly_wait(10)
        print("✅ Navigateur initialisé avec succès")
        return driver
    except Exception as e:
        print(f"❌ Erreur d'initialisation du navigateur: {str(e)}")
        return None

def wait_for_page_ready(driver, timeout=30):
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        print("✅ Page complètement chargée")
        return True
    except TimeoutException:
        print("⚠️ Timeout lors du chargement de la page")
        return False


In [ ]:
pdf_test_path = r"C:\\Users\\zizou\\OneDrive\\Desktop\\stage 3ème\\day 2\\pdffiles\\dfeff758-7b67-4dc4-a77e-2333e405d18a.pdf"
extraire_date(pdf_test_path)

In [ ]:
def extraire_tableaux(pdf_path, debut_section, fin_section):
    """Extrait les tableaux entre deux sections spécifiques"""
    with open(pdf_path, 'rb') as file:
        reader = PdfReader(file)
        texte_complet = ""
        for page in reader.pages:
            texte_complet += page.extract_text() + "\n"

        start_idx = texte_complet.find(debut_section)
        end_idx = texte_complet.find(fin_section)

        if start_idx == -1 or end_idx == -1:
            return []

        section_texte = texte_complet[start_idx + len(debut_section):end_idx]
        lignes = [ligne.strip() for ligne in section_texte.split('\n') if ligne.strip()]

        tableaux = []
        tableau_actuel = []
        for ligne in lignes:
            if re.match(r'^.*\s{2,}.*$', ligne):
                tableau_actuel.append(ligne)
            elif tableau_actuel:
                tableaux.append(tableau_actuel)
                tableau_actuel = []

        if tableau_actuel:
            tableaux.append(tableau_actuel)

        return tableaux

In [5]:
tableaux = extraire_tableaux(
    pdf_test_path,
    "Les opérations de Mise en Pension du jour",
    "Les opérations de Rétrocession des Pensions Livrées"
)
for ligne in tableaux[0]:
    print(ligne)

Montant Pension Livrée en MDT :    178,90 MDT
ISIN Libelle  Nombre de Titres  Montant  Echéance  Taux
TNOXRGYG8RL8         BTA 8% 24 Novembre 2025    1 907     2,001     7    6,750
TNOXRGYG8RL8         BTA 8% 24 Novembre 2025    3 813     4,000     91    7,500
TNUWXR58DVH5         EMP NAT 2024 T1 CB TF    30 000     3,000     10    8,750
TN0008000739         BTA 7,4% Fevrier 2030    1 943     2,000     8    8,500
TN0008000739         BTA 7,4% Fevrier 2030    31 579     30,000     91    8,500
TN0008000739         BTA 7,4% Fevrier 2030    31 579     30,000     120    8,550
TN0008000739         BTA 7,4% Fevrier 2030    1 052     0,999     91    8,500
TNUWXR58DVH5         EMP NAT 2024 T1 CB TF    13 870     1,450     7    8,500
TNUWXR58DVH5         EMP NAT 2024 T1 CB TF    191 299     20,000     14    8,500
TNUWXR58DVH5         EMP NAT 2024 T1 CB TF    191 299     20,000     14    8,500
TNUWXR58DVH5         EMP NAT 2024 T1 CB TF    114 779     12,000     14    8,500
TNUWXR58DVH5         EM

In [6]:
def convertir_en_dataframe(tableau):
    """Convertit un tableau texte en DataFrame"""
    lignes_propres = []
    for ligne in tableau:
        ligne_propre = re.sub(r'\s{2,}', '|', ligne.strip())
        colonnes = ligne_propre.split('|')
        lignes_propres.append(colonnes)

    if len(lignes_propres) < 3:
        print("Tableau trop court après exclusion des lignes 0 et 1, ignoré.")
        return None

    print("Lignes propres extraites :")
    for i, ligne in enumerate(lignes_propres):
        print(f"Ligne {i}: {ligne} ({len(ligne)} colonnes)")

    header = ["ISIN", "Libellé", "Nombre de Titres", "Montant", "Échéance", "Taux"]
    lignes_normalisees = []
    for ligne in lignes_propres[2:]:
        if len(ligne) < 6:
            ligne = ligne + [''] * (6 - len(ligne))
        elif len(ligne) > 6:
            ligne = ligne[:6]
        lignes_normalisees.append(ligne)

    if not lignes_normalisees:
        print("Aucune donnée valide après exclusion des lignes 0 et 1.")
        return None

    try:
        df = pd.DataFrame(lignes_normalisees, columns=header)
        print(f"DataFrame créé avec {len(df)} lignes")
        return df
    except Exception as e:
        print(f"Erreur DataFrame : {str(e)}")
        return None

In [9]:
df = convertir_en_dataframe(tableaux[0])
df

Lignes propres extraites :
Ligne 0: ['Montant Pension Livrée en MDT :', '178,90 MDT'] (2 colonnes)
Ligne 1: ['ISIN Libelle', 'Nombre de Titres', 'Montant', 'Echéance', 'Taux'] (5 colonnes)
Ligne 2: ['TNOXRGYG8RL8', 'BTA 8% 24 Novembre 2025', '1 907', '2,001', '7', '6,750'] (6 colonnes)
Ligne 3: ['TNOXRGYG8RL8', 'BTA 8% 24 Novembre 2025', '3 813', '4,000', '91', '7,500'] (6 colonnes)
Ligne 4: ['TNUWXR58DVH5', 'EMP NAT 2024 T1 CB TF', '30 000', '3,000', '10', '8,750'] (6 colonnes)
Ligne 5: ['TN0008000739', 'BTA 7,4% Fevrier 2030', '1 943', '2,000', '8', '8,500'] (6 colonnes)
Ligne 6: ['TN0008000739', 'BTA 7,4% Fevrier 2030', '31 579', '30,000', '91', '8,500'] (6 colonnes)
Ligne 7: ['TN0008000739', 'BTA 7,4% Fevrier 2030', '31 579', '30,000', '120', '8,550'] (6 colonnes)
Ligne 8: ['TN0008000739', 'BTA 7,4% Fevrier 2030', '1 052', '0,999', '91', '8,500'] (6 colonnes)
Ligne 9: ['TNUWXR58DVH5', 'EMP NAT 2024 T1 CB TF', '13 870', '1,450', '7', '8,500'] (6 colonnes)
Ligne 10: ['TNUWXR58DVH5', 

,ISIN,Libellé,Nombre de Titres,Montant,Échéance,Taux
0,TNOXRGYG8RL8,BTA 8% 24 Novembre 2025,1 907,"2,001",7,"6,750"
1,TNOXRGYG8RL8,BTA 8% 24 Novembre 2025,3 813,"4,000",91,"7,500"
2,TNUWXR58DVH5,EMP NAT 2024 T1 CB TF,30 000,"3,000",10,"8,750"
3,TN0008000739,"BTA 7,4% Fevrier 2030",1 943,"2,000",8,"8,500"
4,TN0008000739,"BTA 7,4% Fevrier 2030",31 579,"30,000",91,"8,500"
5,TN0008000739,"BTA 7,4% Fevrier 2030",31 579,"30,000",120,"8,550"
6,TN0008000739,"BTA 7,4% Fevrier 2030",1 052,"0,999",91,"8,500"
7,TNUWXR58DVH5,EMP NAT 2024 T1 CB TF,13 870,"1,450",7,"8,500"
8,TNUWXR58DVH5,EMP NAT 2024 T1 CB TF,191 299,"20,000",14,"8,500"
9,TNUWXR58DVH5,EMP NAT 2024 T1 CB TF,191 299,"20,000",14,"8,500"
